#체크포인트

* LangGraph는 상태(State) 중심으로 동작하는 워크플로우 엔진입니다.
* 실행 도중의 상태(state)를 체크포인트로 저장해두면, 예외가 발생하거나 시스템이 중단되어도 저장된(체크포인트) 시점부터 이어서 실행할 수 있습니다.



LangGraph 체크포인트의 동작 방식

* LangGraph의 체크포인트 시스템은 세 가지 핵심 구성요소로 작동합니다:
  1. StateGraph : 그래프의 구조(노드, 에지, 상태)를 정의하는 객체
  2. CheckPointer: 체크포인트를 실제로 저장/복원하는 역할 담당(MemorySaver, SqliteSaver, Postgreeaver등)
    * 단, MemorySaver : 프로그램 재시작 시 체크포인트가 사라지는 한계가 있습니다.
  3. RunnableConfig : 실행 세션(예:thread_id)을 구분하는 설정 객체 - 체크포인트 식별에 이용

실행 흐름 요약

1. 워크플로우 시작 시
   * checkpointer.get_tuple(config)을 호출해 이전 상태가 있는지 확인
   * 있으면 graph.set_tuple(checkpoint_tuple)로 복원
   * 없으면 새로운 워크플로우를 시작
2. 그래프 실행 중
   * 각 노드 실행이 완료될 때마다 LangGraph가 내부적으로 상태를 저장
   * Checkpointer가 해당 상태를 thread_id 기반으로 기록
3. 오류 발생 시
   * 예외가 발생해도, 이전 체크포인트 상태는 안전하게 저장되어 있음
   * 이후 get_tuple(config)을 호출하면 그 시점의 상태를 복원 가능
4. 재개 시
   * graph.set_tuple(checkpoint_tuple)으로 상태를 복원하고,
   * invoke()를 다시 호출하면 중단된 곳부터 실행 재개

##기본 설정


In [3]:
import os
from google.colab import userdata

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "chekpoint-test"
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY");


In [4]:
!pip install langchain>=0.3.0 langchain-openai>=0.2.0  langgraph==1.0.2

In [ ]:
!pip install langgraph-checkpoint-sqlite

In [6]:
# Version : 3.0.0
!pip show langgraph-checkpoint-sqlite

Name: langgraph-checkpoint-sqlite
Version: 3.0.0
Summary: Library with a SQLite implementation of LangGraph checkpoint saver.
Home-page: 
Author: 
Author-email: 
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: aiosqlite, langgraph-checkpoint, sqlite-vec
Required-by: 


##그래프 스테이트와 노드 함수 정의

* 체크포인트 작동을 확인하는 것이 목적이므로 OpenAI API를 이용한 간단한 대화 에이전트를 만듭니다.

In [29]:
import operator
from typing import Any, Annotated
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field


# 그래프의 스테이트 정의
class State(BaseModel):
  query: str = Field(default="", description="사용자의 질문")
  messages: Annotated[list[BaseMessage], operator.add] = Field(default=[], description="사용자와 LLM 간의 대화 기록")


# 메시지를 추가하는 노드 함수
def add_message(state: State) -> dict[str, Any]:

  additional_messages = []

  if not state.messages:
      additional_messages.append(SystemMessage(content="당신은 최소한의 응답을 하는 대화 에이전트입니다."))

  additional_messages.append(HumanMessage(content=state.query))

  return {"messages": additional_messages}


# LLM 응답을 추가하는 노드 함수
def llm_response(state: State) -> dict[str, Any]:
  print("LLM 응답 노드 실행 중...")

  # -------------------------------------------------------------------------------
  #                    테스트용 예외 발생
  # -------------------------------------------------------------------------------
  if "오류" in state.query:
      raise RuntimeError("사용자 입력에 '오류'가 포함되어 있어 강제 중단!")

  model = ChatOpenAI(
      model="gpt-4o-mini",
      temperature=0.5,
      openai_api_key=os.environ["OPENAI_API_KEY"])

  ai_message = model.invoke(state.messages)  # 현재까지의 대화(state.messages)를 입력으로 LLM에 전달하여 답변 생성

  return {"messages": [ai_message]}



##워크플로우 실행 함수 (그래프 정의 및 컴파일)

SqliteSaver
* SQLite 데이터베이스에 체크포인트 정보를 저장하는 체크포인터
* MemorySaver와 달리 체크포인터 정보가 영속화됨.
* 프로덕션 시스템에서 사용하는 것 권장되지 않지만 소규모 시스템 운영에는 충분히 사용 가능.

In [30]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.base import BaseCheckpointSaver
from langchain_core.runnables import RunnableConfig


def run_workflow(state: State, checkpoint_file: str, config: RunnableConfig):

    try:
        checkpoint_dir = os.path.dirname(checkpoint_file)

        # 상위 디렉토리 없으면 생성
        os.makedirs(os.path.dirname(checkpoint_file), exist_ok=True)

        if os.path.exists(checkpoint_dir) and os.path.isdir(checkpoint_dir):
            print(f"디렉토리가 존재합니다: {checkpoint_dir}")
        else:
            print(f"디렉토리가 존재하지 않습니다: {checkpoint_dir}")

        #  쓰기 권한 확인
        if os.access(os.path.dirname(checkpoint_file), os.W_OK):
            print("쓰기 권한 있음")
        else:
            print("쓰기 권한 없음")


        ## SQLite CheckPointer 생성
        with SqliteSaver.from_conn_string(checkpoint_file) as checkpointer:

            graph = StateGraph(state_schema=State)

            # 노드 추가
            graph.set_entry_point("add_message")
            graph.add_node(add_message, name="add_message")
            graph.add_node(llm_response, name="llm_response")

            # 노드 연결
            graph.add_edge("add_message", "llm_response")
            graph.add_edge("llm_response", END)

            # 체크포인트 확인
            # 지정된 설정을 사용해 CheckpointTuple을 가져온다.
            checkpoint_tuple = checkpointer.get_tuple(config)
            print("CheckpointTuple : ", checkpoint_tuple)

            if checkpoint_tuple is not None:
                print('이전 상태 재개')
            else:
                print('새 워크플로우 시작')

            # 그래프 실행
            # checkpoint_tuple is not None 인 경우도 이전 상태를 재개한다.
            compiled_graph = graph.compile(checkpointer=checkpointer)

            result = compiled_graph.invoke(state, config=config)
            print("워크플로우 수행 결과:", result)


    except Exception as e:
        print("오류 발생! 마지막 체크포인트에서 재개 가능")
        raise e



## 실행 - 작동 확인하기

In [32]:
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, END
from langchain_core.runnables import RunnableConfig
import os


# 체크포인트 저장용 SQLite 파일 경로
CHECKPOINT_FILE = "/content/checkpoints/test2.db"

checkpointer = SqliteSaver.from_conn_string(CHECKPOINT_FILE)

# 그래프 간의 실행 세션을 구분하기 위해서 설정
config = RunnableConfig(configurable={"thread_id": "example-1"})

state = State(query="제가 좋아하는 것은 찰떡입니다. 기억해 주세요")

# -------------------------------------------------------------------------------
#                    테스트용 예외 발생
# -------------------------------------------------------------------------------
# state = State(query="오류")

run_workflow(state, CHECKPOINT_FILE, config)


디렉토리가 존재합니다: /content/checkpoints
쓰기 권한 있음
CheckpointTuple :  CheckpointTuple(config={'configurable': {'thread_id': 'example-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c918c-a151-6f06-800c-0997810ac557'}}, checkpoint={'v': 4, 'ts': '2025-11-24T09:34:35.526517+00:00', 'id': '1f0c918c-a151-6f06-800c-0997810ac557', 'channel_values': {'query': '오류', 'messages': [SystemMessage(content='당신은 최소한의 응답을 하는 대화 에이전트입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='제가 좋아하는 것은 찰떡입니다. 기억해 주세요', additional_kwargs={}, response_metadata={}), AIMessage(content='알겠습니다! 찰떡을 좋아하시는군요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 42, 'total_tokens': 56, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-202

##체크포인트 내용을 표시하는 함수
* CheckerPointer(SqliteSaver)에서 최신 CheckpointTuple을 이용하여 CheckPpoint 정보와 CheckpointMetadata 정보를 표시한다.
* 체크포인트 정보가 어떻게 변화하는지 모니터링 할 수 있다.

In [33]:
from pprint import pprint
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.base import BaseCheckpointSaver

# BaseCheckpointSaver : 추상 클래스로 실제 구현체(SqliteSaver)를 사용합니다.
def print_checkpoint_dump(checkpoint_file: str, config: RunnableConfig):
    # context manager 안에서만 checkpointer 사용 가능
    with SqliteSaver.from_conn_string(checkpoint_file) as checkpointer:
        checkpoint_tuple = checkpointer.get_tuple(config)
        print("체크포인트 데이터")
        pprint(checkpoint_tuple.checkpoint if checkpoint_tuple else None)
        print("\n메타데이터")
        pprint(checkpoint_tuple.metadata if checkpoint_tuple else None)



최신 체크포인트의 상세 데이터 조회
* channel_values : 체크포인트 시점의 state 값
* ts: 체크포인트가 생성된 시각(ISO 8601형식) : UTC



In [34]:
print_checkpoint_dump(CHECKPOINT_FILE, config)

체크포인트 데이터
{'channel_values': {'messages': [SystemMessage(content='당신은 최소한의 응답을 하는 대화 에이전트입니다.', additional_kwargs={}, response_metadata={}),
                                 HumanMessage(content='제가 좋아하는 것은 찰떡입니다. 기억해 주세요', additional_kwargs={}, response_metadata={}),
                                 AIMessage(content='알겠습니다! 찰떡을 좋아하시는군요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 42, 'total_tokens': 56, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CfMj3BFg1RGSkcbdBcpl1rm3ksC9I', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--3c3c7d1d-4226-49fb-8afc-2ef9b238502d-0', usage_metadata={'input_tokens': 42

 checkpointer(SqliteSaver)에 저장된 특정 실행 세션(thread_id="example-1")에 해당하는 체크포인트(즉, 그래프 실행 중간 상태) 목록 조회
 1. 입력 수신
 2. 메시지 추가 노드
 3. LLM 응답 노드
 4. LLM 출력 반영

 input → add_message → llm_response → finalize
 총 4개의 CheckpointTuple이 생성됩니다.

In [35]:

# checkpointer(SqliteSaver)에 저장된 특정 실행 세션(thread_id="example-1")에 해당하는 체크포인트(즉, 그래프 실행 중간 상태)들을 리스트 형태로 조회할 수 있습니다.
from langgraph.checkpoint.sqlite import SqliteSaver
from typing import Final

with SqliteSaver.from_conn_string(CHECKPOINT_FILE ) as checkpointer:
    for checkpoint in checkpointer.list(config):
        print(checkpoint)

CheckpointTuple(config={'configurable': {'thread_id': 'example-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c918e-1f63-66cf-8010-f485b7a5eb30'}}, checkpoint={'v': 4, 'ts': '2025-11-24T09:35:15.589280+00:00', 'id': '1f0c918e-1f63-66cf-8010-f485b7a5eb30', 'channel_values': {'query': '제가 좋아하는 것은 찰떡입니다. 기억해 주세요', 'messages': [SystemMessage(content='당신은 최소한의 응답을 하는 대화 에이전트입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='제가 좋아하는 것은 찰떡입니다. 기억해 주세요', additional_kwargs={}, response_metadata={}), AIMessage(content='알겠습니다! 찰떡을 좋아하시는군요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 42, 'total_tokens': 56, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560

##실행해 작동 확인하기 (세션 변경)

In [15]:

# 체크포인트 저장용 SQLite 파일 경로

checkpointer = SqliteSaver.from_conn_string(CHECKPOINT_FILE)

config = RunnableConfig(configurable={"thread_id": "example-2"})   # 그래프 간의 실행 세션을 구분하기 위해서 설정

state = State(query="제가 좋아하는 것은 찰떡입니다. 기억해 주세요")

run_workflow(state, CHECKPOINT_FILE, config)


디렉토리가 존재합니다: /content/checkpoints
쓰기 권한 있음
CheckpointTuple :  None
새 워크플로우 시작
LLM 응답 노드 실행 중...
워크플로우 수행 결과: {'query': '제가 좋아하는 것은 찰떡입니다. 기억해 주세요', 'messages': [SystemMessage(content='당신은 최소한의 응답을 하는 대화 에이전트입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='제가 좋아하는 것은 찰떡입니다. 기억해 주세요', additional_kwargs={}, response_metadata={}), AIMessage(content='알겠습니다! 찰떡을 좋아하시는군요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 42, 'total_tokens': 56, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CfN6KGTppSnBk39pfEmHR9IBLbHGD', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--cc609b84-b580-4980-8493

In [16]:
print_checkpoint_dump(CHECKPOINT_FILE, config)

체크포인트 데이터
{'channel_values': {'messages': [SystemMessage(content='당신은 최소한의 응답을 하는 대화 에이전트입니다.', additional_kwargs={}, response_metadata={}),
                                 HumanMessage(content='제가 좋아하는 것은 찰떡입니다. 기억해 주세요', additional_kwargs={}, response_metadata={}),
                                 AIMessage(content='알겠습니다! 찰떡을 좋아하시는군요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 42, 'total_tokens': 56, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CfN6KGTppSnBk39pfEmHR9IBLbHGD', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--cc609b84-b580-4980-8493-e1700ca9d263-0', usage_metadata={'input_tokens': 42

## db 파일 내용 확인

* Colab에서 작업할 때 /content/ 폴더는 세션이 종료되면 초기화됩니다.
* SQLite format 3로 시작하는 첫 몇 바이트는 SQLite 데이터베이스 파일임을 나타내는 시그니처입니다.



In [17]:
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "rb") as f:
        data = f.read(100)  # 처음 100바이트 확인
        print(data)

b'SQLite format 3\x00\x10\x00\x02\x02\x00@  \x00\x00\x00\x03\x00\x00\x00\x08\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x02\x00\x00\x00\x04\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x03\x00.WJ'
